# Spindle — Chaos & Simulation

**v1.3.0** introduces three simulation modes and a chaos engine for testing real-world data pipeline resilience:

| Module | Purpose |
|--------|---------|
| `ChaosEngine` | Inject schema drift, value corruption, late arrivals, FK breaks, volume spikes |
| `FileDropSimulator` | Simulate upstream batch files landing in Lakehouse partitions |
| `StreamEmitter` | Emit rows as CloudEvents envelopes at configurable rates |
| `HybridSimulator` | Run file-drop + stream concurrently, linked by correlation ID |

This notebook walks through each in sequence.

In [1]:
%pip install sqllocks-spindle --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\suref\OneDrive\VSCode\AzureClients\forge-workspace\projects\fabric-datagen\.venv-win\Scripts\python.exe -m pip install --upgrade pip


In [2]:
from sqllocks_spindle import Spindle, RetailDomain

result = Spindle().generate(domain=RetailDomain(), scale='fabric_demo', seed=42)
print(result)
orders = result['order'].copy()
print(f'\nOrder table: {len(orders):,} rows, {len(orders.columns)} columns')

GenerationResult(9 tables, 4,670 total rows, 0.2s)

Order table: 1,000 rows, 8 columns


## 1. Chaos Engine

The `ChaosEngine` injects realistic data quality issues. Configure it once, then call it day-by-day to simulate a 30-day pipeline run with escalating problems.

In [3]:
from sqllocks_spindle.chaos.config import ChaosConfig, ChaosCategory, ChaosOverride
from sqllocks_spindle.chaos.engine import ChaosEngine

cfg = ChaosConfig(
    enabled=True,
    intensity='stormy',      # 2.5x probability multiplier
    seed=99,
    warmup_days=7,           # pipeline runs clean for first 7 days
    chaos_start_day=8,
    escalation='gradual',    # ramps up over 30 days
    breaking_change_day=20,  # column drops/renames only after day 20
)

engine = ChaosEngine(cfg)
print('Intensity multiplier:', cfg.intensity_multiplier)

Intensity multiplier: 2.5


In [4]:
# Show day-by-day chaos decisions across a 30-day window
import pandas as pd

rows = []
for day in range(1, 31):
    row = {'day': day}
    for cat in ChaosCategory:
        row[cat.value] = engine.should_inject(day, cat.value)
    rows.append(row)

chaos_schedule = pd.DataFrame(rows).set_index('day')
# Show only days where something fires
chaos_schedule[chaos_schedule.any(axis=1)]

,schema,value,file,referential,temporal,volume
day,,,,,,
9,False,False,False,True,False,False
13,False,False,False,False,False,True
19,False,False,False,False,False,True
20,False,True,False,False,False,False
22,False,True,False,False,False,False
25,False,False,False,True,False,False
26,False,False,False,False,False,True
27,False,False,False,False,True,False
29,False,True,False,False,False,False


In [5]:
# Value corruption: nulls, out-of-range values, type coercions
corrupted = engine.corrupt_dataframe(orders.copy(), day=15)

print(f'Nulls before: {orders.isnull().sum().sum():,}')
print(f'Nulls after:  {corrupted.isnull().sum().sum():,}')
corrupted.head(3)

Nulls before: 919
Nulls after:  919


,order_id,customer_id,store_id,shipping_address_id,promotion_id,order_date,status,order_total
0,1,1,---,254,143,2025-08-12 20:36:31.081033514,completed,156.40
1,2,10,46,None,None,2025-08-19 12:01:03.000000000,completed,76.20
2,3,24,1,264,None,2037-03-25 00:00:00.000000000,shipped,57.96


In [6]:
# Schema drift: add/remove/rename columns (day 22 is post breaking_change_day)
drifted = engine.drift_schema(orders.copy(), day=22)

added   = [c for c in drifted.columns if c not in orders.columns]
removed = [c for c in orders.columns  if c not in drifted.columns]
print(f'Columns added:   {added}')
print(f'Columns removed: {removed}')
drifted.head(3)

Columns added:   ['status_RENAMED']
Columns removed: ['status', 'order_total']


,order_id,customer_id,store_id,shipping_address_id,promotion_id,order_date,status_RENAMED
0,1,1,101,254,143,2025-08-12 20:36:31.081033514,completed
1,2,10,46,None,None,2025-08-19 12:01:03.000000000,completed
2,3,24,1,264,None,2025-03-26 07:29:42.000000000,shipped


In [7]:
# Volume chaos: spike (10x), empty batch, or single-row batch
date_cols = [c for c in orders.columns if 'date' in c.lower()]

volumed = engine.inject_volume_chaos(orders.copy(), day=10)
print(f'Rows before: {len(orders):,}')
print(f'Rows after:  {len(volumed):,}')

Rows before: 1,000
Rows after:  1


## 2. File-Drop Simulation

`FileDropSimulator` writes data into a partition layout that mirrors what Fabric Data Factory pipelines produce in the Lakehouse Files section.

In [8]:
from sqllocks_spindle.simulation.file_drop import FileDropConfig, FileDropSimulator

cfg_fd = FileDropConfig(
    domain='retail',
    base_path='/lakehouse/default/Files/landing',  # or local path when running outside Fabric
    cadence='daily',
    date_range_start='2025-01-01',
    date_range_end='2025-01-07',
    partitioning='dt=YYYY-MM-DD',
    formats=['parquet'],
    entities=['customer', 'order'],
    manifest_enabled=True,
    done_flag_enabled=True,
    lateness_enabled=True,
    lateness_probability=0.15,
    max_days_late=2,
    duplicates_enabled=True,
    duplicate_probability=0.05,
    seed=42,
)

sim = FileDropSimulator(tables=result.tables, config=cfg_fd)
fd_result = sim.run()

print(f'Files written: {len(fd_result.files_written)}')
print(f'Manifests:     {len(fd_result.manifest_paths)}')
print(f'Done flags:    {len(fd_result.done_flag_paths)}')

Files written: 6
Manifests:     5
Done flags:    5


In [9]:
# Per-entity stats
stats_df = pd.DataFrame([
    {'entity': k, **v}
    for k, v in fd_result.stats.items()
    if isinstance(v, dict) and 'files' in v
])
stats_df

,entity,files,rows_written,formats
0,customer,1,1,[parquet]
1,order,5,4,[parquet]


## 3. Stream Emitter

`StreamEmitter` wraps each row in a CloudEvents-style `EventEnvelope` and emits to a console, file, or real Event Hub sink.

In [10]:
from sqllocks_spindle.simulation.stream_emit import StreamEmitConfig, StreamEmitter

cfg_stream = StreamEmitConfig(
    rate_per_sec=5000,          # no sleeping (realtime=False)
    jitter_ms=10,
    out_of_order_probability=0.05,
    replay_enabled=True,
    replay_window_minutes=3,
    replay_probability=0.10,
    replay_burst_size=5,
    topics=['retail.orders'],
    envelope_schema_version='1.0',
    sink_type='file',
    sink_connection={'path': '/lakehouse/default/Files/stream/orders.jsonl'},
    max_events=500,
    realtime=False,
    seed=42,
)

stream_result = StreamEmitter(
    tables={'order': result['order']},
    config=cfg_stream,
).emit()

print(f'Primary events:  {stream_result.events_sent:,}')
print(f'Replayed events: {stream_result.replay_events_sent:,}')
print(f'Total emitted:   {stream_result.total_events:,}')
print(f'Topics:          {stream_result.topics_used}')

Primary events:  500
Replayed events: 181
Total emitted:   681
Topics:          {'retail.orders'}


## 4. Hybrid Simulation (Batch + Stream)

`HybridSimulator` runs both paths simultaneously, stamping both outputs with the same `correlation_id` so you can join batch manifests to stream event metadata in KQL or Spark.

In [11]:
from sqllocks_spindle.simulation.hybrid import HybridConfig, HybridSimulator
from sqllocks_spindle.simulation.file_drop import FileDropConfig
from sqllocks_spindle.simulation.stream_emit import StreamEmitConfig

cfg_hybrid = HybridConfig(
    stream_to='eventhouse',
    micro_batch_to='lakehouse_files',
    stream_tables=['order'],
    batch_tables=['customer', 'order_line'],
    file_drop_config=FileDropConfig(
        domain='retail',
        base_path='/lakehouse/default/Files/landing',
        cadence='daily',
        date_range_start='2025-01-01',
        date_range_end='2025-01-03',
        formats=['parquet'],
        manifest_enabled=True,
        done_flag_enabled=True,
    ),
    stream_config=StreamEmitConfig(
        rate_per_sec=5000,
        max_events=300,
        sink_type='file',
        sink_connection={'path': '/lakehouse/default/Files/stream/hybrid_orders.jsonl'},
    ),
    link_strategy='correlation_id',
    concurrent=True,
    seed=42,
)

hybrid_result = HybridSimulator(tables=result.tables, config=cfg_hybrid).run()

batch_files = len(hybrid_result.file_drop_result.files_written) if hybrid_result.file_drop_result else 0
stream_evts  = hybrid_result.stream_result.events_sent if hybrid_result.stream_result else 0

print(f'Batch files:    {batch_files}')
print(f'Stream events:  {stream_evts:,}')
print(f'Correlation ID: {hybrid_result.correlation_id}')
print(f'Link strategy:  {hybrid_result.link_strategy}')

Batch files:    13
Stream events:  300
Correlation ID: 9c039096-3211-4880-a682-183ef9165013
Link strategy:  correlation_id


## 5. Chaos + File-Drop combined

The canonical chaos testing pattern: generate data, apply chaos day-by-day, feed the corrupted tables into a file-drop simulation. This replicates what happens when your upstream vendor starts sending bad data.

In [12]:
import copy

# Start with clean tables, corrupt them with escalating chaos
chaos_cfg = ChaosConfig(enabled=True, intensity='stormy', seed=99, warmup_days=3)
chaos_eng = ChaosEngine(chaos_cfg)

corrupted_tables = copy.deepcopy(result.tables)
corrupted_tables['order'] = chaos_eng.apply_all(
    df=result['order'].copy(),
    day=14,
    tables_dict=result.tables,
    date_columns=date_cols if date_cols else None,
)

# Feed corrupted tables into file-drop
chaos_fd = FileDropConfig(
    domain='retail',
    base_path='/lakehouse/default/Files/landing/chaos_test',
    cadence='daily',
    date_range_start='2025-01-14',
    date_range_end='2025-01-14',
    formats=['parquet'],
    entities=['order'],
    manifest_enabled=True,
)

chaos_fd_result = FileDropSimulator(tables=corrupted_tables, config=chaos_fd).run()
print(f'Chaos run: {len(chaos_fd_result.files_written)} file(s) written with corrupted data')
print('Ready to test your Fabric pipeline error handling!')

Chaos run: 0 file(s) written with corrupted data
Ready to test your Fabric pipeline error handling!
